# PhoBERT-base-v2 - 5-fold cross-validation + ensemble (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. The code and processed data are cloned from GitHub (push your latest changes first).

The train and val splits are pooled (3,067 articles) and divided into 5 stratified folds. Five models are trained, each validated on a different fold. This gives:

- an **out-of-fold** score over all 3,067 articles - a much more stable estimate than one 341-row split
- an **ensemble**: the five models' probabilities averaged on the untouched test split

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phobert_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phobert_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/phobert_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "phobert_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train the folds

Each entry in `RUNS` is one full 5-fold run with its own output folder. All runs use the same folds (same `seed`), so their out-of-fold scores are directly comparable.

- `short`: 4 epochs, checkpoint chosen on validation loss (the first run overfitted after epoch 2-3)
- `regularised`: the same plus R-Drop and layer-wise learning rate decay

Checkpoints are deleted after each fold unless `keep_checkpoints=True` (each is ~540 MB).

In [ ]:
from trainer.train_phobert_cv import CVConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    max_length=128,
    use_lead=True,        # False = title only
    batch_size=32,
    lr=2e-5,
    weighted_loss=True,
    patience=3,
    seed=42,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "short": dict(epochs=4, monitor="loss"),
    "regularised": dict(epochs=5, monitor="loss", rdrop_alpha=1.0, layer_decay=0.9),
    # "first_run": dict(epochs=10, monitor="f1"),   # the original settings
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(CVConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare runs

Judge by `oof_f1` (3,067 articles). The test columns rest on 341 articles and move by a point or two from noise alone.

In [ ]:
import pandas as pd

# first run (10 epochs, checkpoint on val F1), for reference
rows = {"first_run": {"oof_f1": 0.7735, "oof_precision": 0.7245, "oof_recall": 0.8297,
                      "single_test_f1": 0.7817, "ensemble_test_f1": 0.7945}}
for name, r in results.items():
    s = r["summary"]
    rows[name] = {"oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
                  "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"]}
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

SHOW = comparison.drop(index="first_run")["oof_f1"].idxmax()   # run detailed below; set a name to override
result = results[SHOW]
print("showing details for:", SHOW)

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
import numpy as np
from trainer.trainer import classification_metrics

oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/phobert_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")